# 2단계 실험: 수집 자료 → RAG 저장 (Chroma)

1단계에서 수집한 `ResearchResult` 의 source content 를 chunking → OpenAI 임베딩 → Chroma 에 영속 저장합니다.

**준비**: `.env` 에 `OPENAI_API_KEY`, `TAVILY_API_KEY`. `data/chroma/` 디렉토리는 자동 생성됩니다.

## 0. 환경 설정 (CWD 를 프로젝트 루트로)

Chroma persist_dir 가 `data/chroma` 상대 경로이므로, 이후 셀들은 프로젝트 루트에서 실행되는 것을 가정합니다.

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    print(f"{k}: {'OK' if os.getenv(k) else 'MISSING'}")
print("CWD:", Path.cwd())

OPENAI_API_KEY: OK
TAVILY_API_KEY: OK
CWD: /Users/jackykim/project/llm-jacky


## 1. 1단계 결과 받아오기

1단계 노트북을 따로 돌릴 필요 없이 여기서 새 주제로 검색합니다.

In [2]:
from llm_jacky.research import run_research

TOPIC = "외국인을 위한 한국 길거리 음식 추천"
result = run_research(TOPIC, max_results=5)
print(f"sources: {len(result.sources)}")
for s in result.sources:
    print(" -", s.title[:60], "|", len(s.content), "chars")

/Users/jackykim/project/llm-jacky/src/llm_jacky/research.py:66: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=max_results)


sources: 5
 - 세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보 | 2199 chars
 - Top 10 Korean Street Foods You Must Try - Love U Korea | 2145 chars
 - [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네 | 1846 chars
 - 외국인이 좋아하는 한식 49선 - 램프쿡 | 2339 chars
 - 우리에겐 일상인데 외국인이 보면 충격받는 한국 길거리 풍경[zum 펌글] : 네이버 블로그 | 2027 chars


## 2. Chunking 미리 보기

Chroma 에 저장하기 전에 어떻게 잘리는지 확인합니다.

In [3]:
from llm_jacky.rag import _to_documents

docs = _to_documents(result)
print(f"chunks: {len(docs)}\n")
for d in docs[:3]:
    print("meta:", {k: v for k, v in d.metadata.items() if k != 'topic'})
    print("text:", d.page_content[:160].replace('\n', ' '), "...\n")

chunks: 16

meta: {'title': '세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보', 'url': 'https://www.christiandaily.co.kr/news/159366', 'source_index': 0, 'chunk_index': 0}
text: ### 1. 태국 — 팟타이와 망고 스티키 라이스  방콕의 골목 어디에서나 “촹촹” 하는 웍 소리가 들리면, 그 옆 포장마차에는 어김없이 팟타이 줄이 서 있다. 쌀국수면을 강한 불에 빠르게 볶고, 새우, 두부, 부추, 숙주, 라임, 땅콩가루를 마지막에 얹어주는 조합은 그 자체로 “세계인 ...

meta: {'title': '세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보', 'url': 'https://www.christiandaily.co.kr/news/159366', 'source_index': 0, 'chunk_index': 1}
text: ### 3. 멕시코 — 타코 알 파스토르 [...] ### 8. 인도 — 파니푸리와 파브 바지  뭄바이의 골목 시장에서는 “파니푸리(Pani puri)”가 한 번에 5~6개씩 입에 쏙쏙 들어간다. 작은 빈 빵 안에 향신료가 들어간 차가운 물과 감자, 콩을 넣어 한입에 톡 깨물어 먹는다. ...

meta: {'title': '세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보', 'url': 'https://www.christiandaily.co.kr/news/159366', 'source_index': 0, 'chunk_index': 2}
text: ### 10. 미국 — 뉴욕 핫도그와 할랄 푸드 트럭 [...] ### 4. 일본 — 타코야키와 오코노미야키  오사카 도톤보리의 밤 풍경에서 절대 빠질 수 없는 것이 바로 타코야키다. 작은 동그란 틀 안에서 문어와 반죽이 굴러가는 모습은 그 자체가 거리 공연 같다. 

## 3. 인덱싱 실행 (Chroma 에 영속 저장)

동일 `(url, chunk_index)` 는 dedupe 됩니다. 다시 실행해도 중복 누적되지 않습니다.

In [4]:
from llm_jacky.rag import index_research

n = index_research(result)
print(f"indexed chunks: {n}")
print("persist dir:", (Path.cwd() / "data" / "chroma").resolve())

indexed chunks: 16
persist dir: /Users/jackykim/project/llm-jacky/data/chroma


## 4. 유사도 검색 (raw)

쿼리에 대해 어떤 chunk 가 검색되는지 확인합니다.

In [5]:
from llm_jacky.rag import _vector_store

vs = _vector_store()
QUERY = "매운 떡볶이를 처음 먹는 외국인에게 어떻게 설명할까?"

hits = vs.similarity_search_with_score(QUERY, k=4)
for doc, score in hits:
    print(f"score={score:.3f} | {doc.metadata.get('title','')[:60]}")
    print(f"  url: {doc.metadata.get('url')}")
    print(f"  text: {doc.page_content[:200].strip()}\n")

score=1.154 | [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네
  url: https://m.blog.naver.com/wjwj38/223176755965
  text: 엘에이 갈비 먹고싶다고 말한건데

​

저날 평소에 먹기 힘든 음식 다 나와서

먹느라 진짜 바빴네욬ㅋㅋㅋㅋ

​

​

제일 귀하고 맛있는 음식 집밥

​

사실은 남편보다 제가 더 좋아하는 음식일지도요~^^

​

한국 과일

​

우리도 해외여행 가면

외국 과일 먹는게 필수 코스잖아요?

​

현지 과일 먹어보는건 외국인들한테도 마찬가지여요!! [.

score=1.225 | [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네
  url: https://m.blog.naver.com/wjwj38/223176755965
  text: 저는 좀 띠용이었어요 ㅋㅋㅋㅋ

생각지도 못했던 불고기버거??

​

​

나름 남편 배려한다고 비싼 버거로 사줬는데

자기껀 안먹고

제가 시킨 불고기 버거 한입먹어보더니 [...] ​

나름 남편 배려한다고 비싼 버거로 사줬는데

자기껀 안먹고

제가 시킨 불고기 버거 한입먹어보더니

순식간에 다 먹어버렸어요

​

제가 맥날을 잘 안가서 몰랐는데요

score=1.241 | 세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보
  url: https://www.christiandaily.co.kr/news/159366
  text: ### 1. 태국 — 팟타이와 망고 스티키 라이스

방콕의 골목 어디에서나 “촹촹” 하는 웍 소리가 들리면, 그 옆 포장마차에는 어김없이 팟타이 줄이 서 있다. 쌀국수면을 강한 불에 빠르게 볶고, 새우, 두부, 부추, 숙주, 라임, 땅콩가루를 마지막에 얹어주는 조합은 그 자체로 “세계인의 입맛 표준”이다. 매콤·새콤·달콤의 균형이 한국인 입맛에도 잘 맞아

s

## 5. Retriever 인터페이스

3단계(Writing Chain) 에서 LCEL 로 연결할 때 그대로 사용할 형태입니다.

In [6]:
from llm_jacky.rag import get_retriever

retriever = get_retriever(k=3)
retrieved = retriever.invoke(QUERY)
for d in retrieved:
    print("-", d.metadata.get("title", "")[:60], "|", d.metadata.get("url"))

- [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네 | https://m.blog.naver.com/wjwj38/223176755965
- [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네 | https://m.blog.naver.com/wjwj38/223176755965
- 세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보 | https://www.christiandaily.co.kr/news/159366


## 6. 컬렉션 상태 / 정리

필요시 컬렉션을 비울 수 있습니다 (실험 리셋용).

In [7]:
vs = _vector_store()
data = vs.get()
print("total docs in collection:", len(data.get("ids", [])))

# 리셋이 필요하면 아래 한 줄 주석 해제:
# vs.delete_collection()

total docs in collection: 16
